# LoDAR-Net + JCAP-Net + ERPD-Net: Corrected Full Implementation
This notebook implements the methodology described in the supplied specification: Indonesian High-School Literary-Criticism Corpus (IHSLCC), hybrid IndoBERT + linguistic/error features, LoRA adaptation, attentive pooling, discourse-move structural features, joint ordinal criterion prediction (C1–C9), holistic ability classification, ERPD evidence–reasoning diagnostics, HDBSCAN weakness-pattern discovery, weakest-criterion analysis, SHAP explainability, counterfactual analysis, and lightweight student-model distillation.

**Important data requirement:** the nine criterion labels and holistic ability labels must come from expert annotation. The notebook does **not** invent ground-truth labels. If the supplied CSV does not contain those labels, training/evaluation cells stop with a clear message and can still run the preprocessing/inference utilities.

In [ ]:
# Install once in a fresh notebook environment
%pip -q install -U pandas numpy scikit-learn scipy statsmodels matplotlib seaborn transformers peft accelerate hdbscan shap sentencepiece openpyxl

In [ ]:
import os, re, time, json, random, warnings, math
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, mean_absolute_error, silhouette_score
from scipy.stats import f_oneway, ttest_rel
import matplotlib.pyplot as plt
import hdbscan
import shap
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from peft import LoraConfig, TaskType, get_peft_model

warnings.filterwarnings("ignore")
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("CUDA:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# =========================
# Configuration
# =========================
DATA_PATH = ""  # Example: "/content/IHSLCC.csv"
TEXT_COL = ""   # Leave empty for automatic detection
ID_COL = ""     # Optional
MODEL_NAME = "indobenchmark/indobert-base-p1"
MAX_LENGTH = 256
BATCH_SIZE = 8 if torch.cuda.is_available() else 4
EPOCHS = 3
LR = 2e-4
ENCODER_LR = 1e-5
WEIGHT_DECAY = 1e-2
WARMUP_RATIO = 0.10
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.10
USE_LORA = True
FREEZE_ENCODER_IF_NO_LORA = True
CRITERION_MIN = 1
CRITERION_MAX = 5
VAL_SIZE = 0.15
TEST_SIZE = 0.15
MIN_CLUSTER_SIZE = 12
TOP_K_EXPLANATIONS = 20
OUT_DIR = Path("IHSLCC_results")
OUT_DIR.mkdir(exist_ok=True)
print("Set DATA_PATH to your labeled IHSLCC CSV before training.")

In [ ]:
# =========================
# Load and harmonize data
# =========================
def load_csv(path):
    if not path:
        candidates = list(Path(".").glob("*.csv")) + list(Path("/content").glob("*.csv"))
        candidates = [p for p in candidates if p.is_file()]
        if not candidates:
            raise FileNotFoundError("No CSV found. Set DATA_PATH to your IHSLCC CSV.")
        path = str(candidates[0])
        print("Auto-selected:", path)
    df = pd.read_csv(path)
    df.columns = [str(c).strip() for c in df.columns]
    return df

df = load_csv(DATA_PATH)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

def find_column(df, candidates):
    low={c.lower().strip():c for c in df.columns}
    for x in candidates:
        if x.lower() in low: return low[x.lower()]
    for c in df.columns:
        cl=c.lower().replace("_"," ").replace("-"," ")
        if any(x.lower() in cl for x in candidates): return c
    return None

if not TEXT_COL:
    TEXT_COL=find_column(df, ["response","student_response","essay","essay_text","text","answer","content","response_text"])
if not TEXT_COL:
    raise ValueError("Could not identify the student-response text column. Set TEXT_COL manually.")
df[TEXT_COL]=df[TEXT_COL].fillna("").astype(str).str.strip()
df=df[df[TEXT_COL].str.len()>0].drop_duplicates(subset=[TEXT_COL]).reset_index(drop=True)
print("Text column:", TEXT_COL)
print("Rows after cleaning:", len(df))

In [ ]:
# =========================
# Identify expert labels
# =========================
CRITERIA=["C1","C2","C3","C4","C5","C6","C7","C8","C9"]
CRITERION_NAMES={
"C1":"Literary Interpretation","C2":"Textual Evidence","C3":"Critical Argument",
"C4":"Analytical Depth","C5":"Literary Terminology","C6":"Organization",
"C7":"Coherence","C8":"Language Accuracy","C9":"Interpretive Originality"
}
def locate_criterion(code):
    exact=[code, code.lower(), f"{code}_score", f"{code}_Score", f"{code} score", f"criterion_{code[1:]}", f"criterion {code[1:]}"]
    return find_column(df, exact)
CRIT_COLS={c:locate_criterion(c) for c in CRITERIA}
ABILITY_COL=find_column(df, ["ability","ability_class","holistic_ability","overall_ability","overall_class","class"])
print("Criterion columns:", CRIT_COLS)
print("Ability column:", ABILITY_COL)
missing=[c for c,v in CRIT_COLS.items() if v is None]
if missing or ABILITY_COL is None:
    print("LABEL CHECK FAILED")
    print("Missing criteria:", missing)
    print("Missing ability:", ABILITY_COL)
    print("Training/evaluation will be blocked until expert labels are supplied.")
else:
    for c,col in CRIT_COLS.items():
        df[col]=pd.to_numeric(df[col], errors="coerce")
    df=df.dropna(subset=list(CRIT_COLS.values())+[ABILITY_COL]).reset_index(drop=True)
    print("Labeled rows:", len(df))

In [ ]:
# =========================
# Text normalization + explicit Indonesian linguistic/error features
# =========================
def normalize_text(x):
    x=str(x)
    x=x.replace("\u200b"," ").replace("\ufeff"," ")
    x=re.sub(r"https?://\S+|www\.\S+"," URL ",x)
    x=re.sub(r"[@#]\w+"," TAG ",x)
    x=re.sub(r"\s+"," ",x).strip()
    return x

df["clean_text"]=df[TEXT_COL].map(normalize_text)

def linguistic_features(text):
    t=str(text)
    words=re.findall(r"\b\w+\b", t.lower(), flags=re.UNICODE)
    sentences=[s for s in re.split(r"[.!?]+", t) if s.strip()]
    n_words=len(words); n_sent=len(sentences)
    unique=len(set(words))
    spelling_like=len(re.findall(r"\b\w*(.)\1{2,}\w*\b", t.lower()))
    punctuation=len(re.findall(r"[,:;.!?]", t))
    uppercase=len(re.findall(r"\b[A-Z]{2,}\b", t))
    digits=len(re.findall(r"\d+", t))
    avg_word=float(np.mean([len(w) for w in words])) if words else 0
    lexical_div=unique/max(n_words,1)
    avg_sent=n_words/max(n_sent,1)
    connectives=sum(len(re.findall(r"\b"+re.escape(w)+r"\b",t.lower())) for w in
                    ["karena","namun","tetapi","sehingga","oleh karena itu","selain itu","meskipun","sedangkan","dengan demikian"])
    evidence_terms=sum(len(re.findall(r"\b"+re.escape(w)+r"\b",t.lower())) for w in
                       ["menurut","dalam teks","kutipan","contoh","bukti","tokoh","novel","cerpen","puisi"])
    interpretation_terms=sum(len(re.findall(r"\b"+re.escape(w)+r"\b",t.lower())) for w in
                       ["menunjukkan","berarti","makna","menggambarkan","menyiratkan","dapat dipahami"])
    return [n_words,n_sent,unique,lexical_div,avg_word,avg_sent,punctuation,uppercase,digits,spelling_like,connectives,evidence_terms,interpretation_terms]
FEAT_NAMES=["word_count","sentence_count","unique_words","lexical_diversity","avg_word_length","avg_sentence_length","punctuation_count","uppercase_count","digit_count","repetition_error","connective_count","evidence_terms","interpretation_terms"]
E=np.asarray([linguistic_features(x) for x in df["clean_text"]],dtype=np.float32)
scaler=StandardScaler()
E=scaler.fit_transform(E).astype(np.float32)
print("Error/linguistic feature shape:", E.shape)

In [ ]:
# =========================
# Discourse-move structural prior
# =========================
# The paper specifies a frozen PERSUADE discourse-move tagger. Because a specific compatible
# Indonesian deployment checkpoint is not supplied here, this notebook uses a transparent
# structural fallback rather than pretending that a PERSUADE model is multilingual.
MOVE_NAMES=["claim","evidence","interpretation","reasoning"]
MOVE_PATTERNS={
"claim":[r"\bmenurut saya\b",r"\bsaya berpendapat\b",r"\bpendapat saya\b",r"\bdapat dikatakan\b"],
"evidence":[r"\bmenurut\b",r"\bberdasarkan\b",r"\bcontoh\b",r"\bbukti\b",r"\bdalam teks\b",r"\bkutipan\b"],
"interpretation":[r"\bmenunjukkan\b",r"\bmakna\b",r"\bberarti\b",r"\bmenggambarkan\b",r"\bmenyiratkan\b"],
"reasoning":[r"\bkarena\b",r"\bsehingga\b",r"\boleh karena itu\b",r"\bdengan demikian\b",r"\bmeskipun\b"]
}
def discourse_vector(text):
    t=text.lower()
    vals=[]
    for move in MOVE_NAMES:
        hits=sum(bool(re.search(p,t)) for p in MOVE_PATTERNS[move])
        vals.append(float(min(hits,3))/3.0)
    # Sequence-order compatibility: claim -> evidence -> interpretation -> reasoning
    pos=[]
    for move in MOVE_NAMES:
        found=[m.start() for p in MOVE_PATTERNS[move] for m in re.finditer(p,t)]
        pos.append(min(found) if found else 10**9)
    order_score=float(sum(pos[i] < pos[i+1] for i in range(3)))/3.0
    vals.append(order_score)
    return vals
D=np.asarray([discourse_vector(x) for x in df["clean_text"]],dtype=np.float32)
print("Discourse feature shape:",D.shape)

In [ ]:
# =========================
# Train/validation/test split
# =========================
if any(v is None for v in CRIT_COLS.values()) or ABILITY_COL is None:
    raise ValueError("Expert C1-C9 and holistic ability labels are required for model training. Add them to the CSV.")
Y=np.column_stack([df[CRIT_COLS[c]].astype(int).clip(CRITERION_MIN,CRITERION_MAX).values for c in CRITERIA])
ability_raw=df[ABILITY_COL].astype(str).str.strip()
ABILITY_MAP={"emerging":0,"developing":1,"proficient":2,"advanced":3}
ability=np.array([ABILITY_MAP.get(x.lower(),np.nan) for x in ability_raw],dtype=float)
if np.isnan(ability).any():
    # Support numeric 0..3 or 1..4 labels.
    numeric=pd.to_numeric(ability_raw,errors="coerce")
    if numeric.notna().all():
        vals=numeric.astype(int).values
        if set(np.unique(vals)).issubset({1,2,3,4}): vals=vals-1
        ability=vals.astype(float)
if np.isnan(ability).any() or not set(np.unique(ability.astype(int))).issubset({0,1,2,3}):
    raise ValueError("Ability labels must be Emerging/Developing/Proficient/Advanced or numeric 0-3/1-4.")
ability=ability.astype(np.int64)

idx=np.arange(len(df))
tr_idx, temp_idx=train_test_split(idx,test_size=VAL_SIZE+TEST_SIZE,random_state=SEED,stratify=ability)
relative_test=TEST_SIZE/(VAL_SIZE+TEST_SIZE)
va_idx,te_idx=train_test_split(temp_idx,test_size=relative_test,random_state=SEED,stratify=ability[temp_idx])
print("Train/Val/Test:",len(tr_idx),len(va_idx),len(te_idx))

In [ ]:
# =========================
# Tokenizer and dataset
# =========================
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)

class EssayDataset(Dataset):
    def __init__(self, indices):
        self.indices=np.asarray(indices)
    def __len__(self): return len(self.indices)
    def __getitem__(self,i):
        j=int(self.indices[i])
        enc=tokenizer(df.loc[j,"clean_text"],max_length=MAX_LENGTH,padding="max_length",truncation=True,return_tensors="pt")
        item={k:v.squeeze(0) for k,v in enc.items()}
        item["error_features"]=torch.tensor(E[j],dtype=torch.float32)
        item["discourse_features"]=torch.tensor(D[j],dtype=torch.float32)
        item["criteria"]=torch.tensor(Y[j]-CRITERION_MIN,dtype=torch.long)
        item["ability"]=torch.tensor(ability[j],dtype=torch.long)
        item["row_index"]=torch.tensor(j,dtype=torch.long)
        return item

train_loader=DataLoader(EssayDataset(tr_idx),batch_size=BATCH_SIZE,shuffle=True,num_workers=0,pin_memory=torch.cuda.is_available())
val_loader=DataLoader(EssayDataset(va_idx),batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
test_loader=DataLoader(EssayDataset(te_idx),batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
print("Batches:",len(train_loader),len(val_loader),len(test_loader))

In [ ]:
# =========================
# LoDAR-Net encoder + JCAP-Net
# =========================
class AttentionPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.score=nn.Sequential(nn.Linear(hidden,hidden//2),nn.Tanh(),nn.Linear(hidden//2,1))
    def forward(self,x,mask):
        s=self.score(x).squeeze(-1)
        s=s.masked_fill(mask==0,-1e4)
        a=torch.softmax(s,dim=1)
        return torch.bmm(a.unsqueeze(1),x).squeeze(1),a

class LoDARJCAP(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder=AutoModel.from_pretrained(MODEL_NAME)
        hidden=self.encoder.config.hidden_size
        if USE_LORA:
            cfg=LoraConfig(task_type=TaskType.FEATURE_EXTRACTION,r=LORA_R,lora_alpha=LORA_ALPHA,
                           lora_dropout=LORA_DROPOUT,target_modules=["query","value"],bias="none")
            self.encoder=get_peft_model(self.encoder,cfg)
            self.encoder.print_trainable_parameters()
        elif FREEZE_ENCODER_IF_NO_LORA:
            for p in self.encoder.parameters(): p.requires_grad=False
        self.pool=AttentionPool(hidden)
        self.proj=nn.Sequential(
            nn.Linear(hidden+len(FEAT_NAMES)+len(MOVE_NAMES)+1,256),
            nn.LayerNorm(256),nn.GELU(),nn.Dropout(.20),
            nn.Linear(256,128),nn.LayerNorm(128),nn.GELU(),nn.Dropout(.15)
        )
        self.criterion_heads=nn.ModuleList([nn.Linear(128,CRITERION_MAX-CRITERION_MIN) for _ in range(9)])
        self.ability_head=nn.Linear(128,4)
    def forward(self,input_ids,attention_mask,error_features,discourse_features):
        out=self.encoder(input_ids=input_ids,attention_mask=attention_mask).last_hidden_state
        pooled,att=self.pool(out,attention_mask)
        x=torch.cat([pooled,error_features,discourse_features],dim=1)
        x=self.proj(x)
        ordinal=[h(x) for h in self.criterion_heads]
        ability_logits=self.ability_head(x)
        return ordinal,ability_logits,x,att

model=LoDARJCAP().to(DEVICE)
print("Model ready.")

In [ ]:
# =========================
# Ordinal loss and training
# =========================
def ordinal_loss(logits,target):
    # K-level ordinal regression represented by K-1 cumulative binary decisions.
    target_bin=torch.stack([(target>k).float() for k in range(CRITERION_MAX-CRITERION_MIN)],dim=1)
    return F.binary_cross_entropy_with_logits(logits,target_bin)

def joint_loss(ord_logits,ability_logits,yc,ya):
    ol=torch.stack([ordinal_loss(o,yc[:,i]) for i,o in enumerate(ord_logits)]).mean()
    al=F.cross_entropy(ability_logits,ya)
    return ol,al,ol+al

trainable=[p for p in model.parameters() if p.requires_grad]
optimizer=torch.optim.AdamW(trainable,lr=LR,weight_decay=WEIGHT_DECAY)
steps=len(train_loader)*EPOCHS
scheduler=get_linear_schedule_with_warmup(optimizer,int(steps*WARMUP_RATIO),steps)

def run_epoch(loader,train=True):
    model.train(train)
    losses=[]; ords=[]; abs_= []; ys=[]; ps=[]; yca=[]; pca=[]
    for b in loader:
        ids=b["input_ids"].to(DEVICE); mask=b["attention_mask"].to(DEVICE)
        ef=b["error_features"].to(DEVICE); dfv=b["discourse_features"].to(DEVICE)
        yc=b["criteria"].to(DEVICE); ya=b["ability"].to(DEVICE)
        with torch.set_grad_enabled(train):
            ol,al,_=model(ids,mask,ef,dfv)
            lo,la,loss=joint_loss(ol,al,yc,ya)
            if train:
                optimizer.zero_grad(set_to_none=True); loss.backward()
                torch.nn.utils.clip_grad_norm_(trainable,1.0); optimizer.step(); scheduler.step()
        losses.append(loss.item())
        pp=[]
        for o in ol:
            # cumulative probabilities P(Y>k); convert to expected score by summing thresholds
            prob=torch.sigmoid(o)
            expected=prob.sum(dim=1)
            pp.append(torch.round(expected).long().clamp(0,CRITERION_MAX-CRITERION_MIN).cpu().numpy())
        yca.append(yc.cpu().numpy()); pca.append(np.stack(pp,axis=1))
        ys.append(ya.cpu().numpy()); ps.append(torch.argmax(al,1).cpu().numpy())
    yca=np.vstack(yca); pca=np.vstack(pca); ys=np.concatenate(ys); ps=np.concatenate(ps)
    return np.mean(losses),yca,pca,ys,ps

history=[]
for epoch in range(1,EPOCHS+1):
    t=time.time()
    loss,_,_,_,_=run_epoch(train_loader,True)
    vloss,vyc,vpc,vya,vpa=run_epoch(val_loader,False)
    history.append([epoch,loss,vloss,accuracy_score(vya,vpa),f1_score(vya,vpa,average="macro")])
    print(f"Epoch {epoch}/{EPOCHS} | train={loss:.4f} | val={vloss:.4f} | ability_acc={history[-1][3]:.4f} | macroF1={history[-1][4]:.4f} | {time.time()-t:.1f}s")
history_df=pd.DataFrame(history,columns=["epoch","train_loss","val_loss","ability_accuracy","ability_macro_f1"])
history_df.to_csv(OUT_DIR/"training_history.csv",index=False)

In [ ]:
# =========================
# Final evaluation
# =========================
test_loss,tyc,tpc,tya,tpa=run_epoch(test_loader,False)
criterion_metrics=[]
for i,c in enumerate(CRITERIA):
    mae=mean_absolute_error(tyc[:,i]+CRITERION_MIN,tpc[:,i]+CRITERION_MIN)
    qwk=cohen_kappa_score(tyc[:,i]+CRITERION_MIN,tpc[:,i]+CRITERION_MIN,weights="quadratic")
    criterion_metrics.append([c,CRITERION_NAMES[c],mae,qwk])
crit_df=pd.DataFrame(criterion_metrics,columns=["code","criterion","MAE","QWK"])
ability_metrics=pd.DataFrame([{
    "accuracy":accuracy_score(tya,tpa),
    "macro_F1":f1_score(tya,tpa,average="macro"),
    "QWK":cohen_kappa_score(tya,tpa,weights="quadratic")
}])
print("Test loss:",test_loss)
display(ability_metrics)
display(crit_df)
crit_df.to_csv(OUT_DIR/"criterion_metrics.csv",index=False)
ability_metrics.to_csv(OUT_DIR/"ability_metrics.csv",index=False)

In [ ]:
# =========================
# ERPD-Net: evidence-argument diagnostic profile + HDBSCAN
# =========================
# Rule-based diagnostic is intentionally transparent. C2-C4 scores are supporting weights,
# not the sole inputs.
def erpd_profile(text,c2,c3,c4):
    dv=np.array(discourse_vector(text))
    claim,evidence,interpretation,reasoning,order=dv
    c2n=c2/(CRITERION_MAX); c3n=c3/(CRITERION_MAX); c4n=c4/(CRITERION_MAX)
    evidence_quality=np.clip(.60*evidence+.40*c2n,0,1)
    argument_quality=np.clip(.55*claim+.45*c3n,0,1)
    interpretation_strength=np.clip(.55*interpretation+.45*c4n,0,1)
    reasoning_strength=np.clip(.50*reasoning+.30*c3n+.20*order,0,1)
    return [evidence_quality,argument_quality,interpretation_strength,reasoning_strength]

texts=df.iloc[te_idx]["clean_text"].tolist()
profiles=[]
for j,row in enumerate(te_idx):
    s=tyc[j]+CRITERION_MIN
    profiles.append(erpd_profile(df.loc[row,"clean_text"],s[1],s[2],s[3]))
profiles=np.asarray(profiles,dtype=np.float32)
# Combine predicted criteria, ability probabilities, error features and ERPD profile.
with torch.no_grad():
    all_lat=[]; all_ap=[]
    for b in test_loader:
        ids=b["input_ids"].to(DEVICE); mask=b["attention_mask"].to(DEVICE)
        ef=b["error_features"].to(DEVICE); dfv=b["discourse_features"].to(DEVICE)
        ol,al,latent,_=model(ids,mask,ef,dfv)
        all_lat.append(latent.cpu().numpy())
        all_ap.append(torch.softmax(al,1).cpu().numpy())
latent=np.vstack(all_lat); ability_prob=np.vstack(all_ap)
cluster_X=np.hstack([tpc+CRITERION_MIN,ability_prob,E[te_idx],profiles])
clusterer=hdbscan.HDBSCAN(min_cluster_size=MIN_CLUSTER_SIZE,min_samples=max(5,MIN_CLUSTER_SIZE//2),prediction_data=True)
labels=clusterer.fit_predict(cluster_X)
valid=labels>=0
n_clusters=len(set(labels[valid])) if valid.any() else 0
noise=float(np.mean(labels<0))
sil=silhouette_score(cluster_X[valid],labels[valid]) if valid.any() and len(set(labels[valid]))>1 else np.nan
print("HDBSCAN clusters:",n_clusters,"noise proportion:",noise,"silhouette:",sil)

cluster_names={}
for k in sorted(set(labels)):
    if k<0: continue
    ids=np.where(labels==k)[0]
    m=np.mean(tpc[ids]+CRITERION_MIN,axis=0)
    weakest=np.argsort(m)[:3]
    if m[7] < np.mean(m)-.35: name="Linguistically Weak"
    elif m[1] < np.mean(m)-.35: name="Evidence Deficient"
    elif m[2] < np.mean(m)-.35 or m[3] < np.mean(m)-.35: name="Reasoning Deficient"
    elif np.mean(m[:4]) < np.mean(m[4:]): name="Analytically Superficial"
    else: name="Descriptive Writer"
    cluster_names[k]=name

pred_df=pd.DataFrame({
"row_index":te_idx,"cluster":labels,
"cluster_profile":[cluster_names.get(x,"Noise") for x in labels],
"evidence_quality":profiles[:,0],"argument_quality":profiles[:,1],
"interpretation_strength":profiles[:,2],"reasoning_strength":profiles[:,3],
"predicted_ability":tpa
})
for i,c in enumerate(CRITERIA): pred_df[f"{c}_pred"]=tpc[:,i]+CRITERION_MIN
pred_df.to_csv(OUT_DIR/"ERPD_HDBSCAN_student_profiles.csv",index=False)
display(pred_df.head())

In [ ]:
# =========================
# Weakest-criteria analysis + ANOVA
# =========================
weak=[]
for i,c in enumerate(CRITERIA):
    vals=tpc[:,i]+CRITERION_MIN
    weak.append([c,CRITERION_NAMES[c],vals.mean(),vals.var(),np.median(vals)])
weak_df=pd.DataFrame(weak,columns=["code","criterion","mean","variance","median"]).sort_values("mean")
display(weak_df)

anova=[]
for i,c in enumerate(CRITERIA):
    groups=[tpc[tpa==a,i]+CRITERION_MIN for a in range(4) if np.sum(tpa==a)>1]
    if len(groups)>=2:
        stat,p=f_oneway(*groups)
    else: stat,p=np.nan,np.nan
    anova.append([c,CRITERION_NAMES[c],stat,p])
anova_df=pd.DataFrame(anova,columns=["code","criterion","F","p_value"])
display(anova_df)
weak_df.to_csv(OUT_DIR/"weakest_criteria.csv",index=False)
anova_df.to_csv(OUT_DIR/"criterion_ANOVA.csv",index=False)

plt.figure(figsize=(10,5))
plt.bar(weak_df["criterion"],weak_df["mean"])
plt.xticks(rotation=45,ha="right"); plt.ylabel("Mean predicted score"); plt.title("Population-Level Weakest Criteria")
plt.tight_layout(); plt.savefig(OUT_DIR/"weakest_criteria.png",dpi=300); plt.show()

In [ ]:
# =========================
# Top-result prediction loop
# =========================
ABILITY_NAMES=["Emerging","Developing","Proficient","Advanced"]
def predict_texts(texts,batch_size=8,return_attention=False):
    model.eval()
    rows=[]
    for start in range(0,len(texts),batch_size):
        chunk=[normalize_text(x) for x in texts[start:start+batch_size]]
        enc=tokenizer(chunk,max_length=MAX_LENGTH,padding="max_length",truncation=True,return_tensors="pt").to(DEVICE)
        ef=np.asarray([linguistic_features(x) for x in chunk],dtype=np.float32)
        ef=scaler.transform(ef).astype(np.float32)
        dv=np.asarray([discourse_vector(x) for x in chunk],dtype=np.float32)
        with torch.no_grad():
            ol,al,_,att=model(enc["input_ids"],enc["attention_mask"],torch.tensor(ef,dtype=torch.float32,device=DEVICE),torch.tensor(dv,dtype=torch.float32,device=DEVICE))
            ap=torch.softmax(al,1).cpu().numpy()
            cp=np.column_stack([torch.round(torch.sigmoid(o).sum(1)).long().clamp(0,4).cpu().numpy()+CRITERION_MIN for o in ol])
        for i,t in enumerate(chunk):
            r={"text":t,"ability":ABILITY_NAMES[int(np.argmax(ap[i]))],"ability_confidence":float(np.max(ap[i]))}
            for j,c in enumerate(CRITERIA): r[f"{c}_score"]=int(cp[i,j])
            r["diagnostic_mean"]=float(np.mean(cp[i]))
            r["weakest_criterion"]=CRITERION_NAMES[CRITERIA[int(np.argmin(cp[i]))]]
            rows.append(r)
    out=pd.DataFrame(rows)
    return out.sort_values(["ability_confidence","diagnostic_mean"],ascending=False).reset_index(drop=True)

# Replace with any new student responses.
new_responses=[
    "Tokoh utama menunjukkan keberanian karena ia tetap mempertahankan prinsipnya meskipun menghadapi konflik.",
    "Menurut saya cerita ini menarik."
]
top_predictions=predict_texts(new_responses)
display(top_predictions)
top_predictions.to_csv(OUT_DIR/"top_predictions.csv",index=False)

In [ ]:
# =========================
# SHAP explainability on the JCAP representation
# =========================
# SHAP is applied to the compact JCAP representation, which is stable and much cheaper
# than explaining every transformer token. We explain the ability head.
class AbilityFromLatent(nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base=base
    def forward(self,x):
        return self.base.ability_head(x)

explainer_model=AbilityFromLatent(model).to(DEVICE).eval()
background=torch.tensor(latent[:min(50,len(latent))],dtype=torch.float32,device=DEVICE)
explain_sample=torch.tensor(latent[:min(TOP_K_EXPLANATIONS,len(latent))],dtype=torch.float32,device=DEVICE)
try:
    shap_explainer=shap.DeepExplainer(explainer_model,background)
    sv=shap_explainer.shap_values(explain_sample)
    print("SHAP explanation generated.")
except Exception as e:
    print("SHAP backend fallback:",e)
    sv=None

if sv is not None:
    arr=sv[0] if isinstance(sv,list) else sv
    if arr.ndim==3: arr=np.mean(np.abs(arr),axis=2)
    mean_abs=np.mean(np.abs(arr),axis=0)
    shap_df=pd.DataFrame({"representation_dimension":np.arange(len(mean_abs)),"mean_abs_SHAP":mean_abs}).sort_values("mean_abs_SHAP",ascending=False)
    shap_df.to_csv(OUT_DIR/"SHAP_representation_importance.csv",index=False)
    display(shap_df.head(20))

In [ ]:
# =========================
# Simple counterfactual analysis
# =========================
# Counterfactual = remove selected structural/error cues and re-run prediction.
def counterfactual_one(text):
    variants={
        "original":text,
        "without_connective_cues":re.sub(r"\b(karena|sehingga|namun|tetapi|oleh karena itu|dengan demikian)\b","",text,flags=re.I),
        "without_evidence_cues":re.sub(r"\b(menurut|berdasarkan|contoh|bukti|dalam teks|kutipan)\b","",text,flags=re.I),
        "shortened": " ".join(text.split()[:max(5,len(text.split())//2)])
    }
    return pd.DataFrame([{ "variant":k, **v} for k,v in
        [(k,predict_texts([v]).iloc[0].to_dict()) for k,v in variants.items()]])
cf=counterfactual_one(new_responses[0])
display(cf[["variant","ability","ability_confidence","diagnostic_mean","weakest_criterion"]])
cf.to_csv(OUT_DIR/"counterfactual_analysis.csv",index=False)

In [ ]:
# =========================
# Knowledge distillation: lightweight student on frozen JCAP representations
# =========================
class Student(nn.Module):
    def __init__(self,in_dim):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(in_dim,96),nn.ReLU(),nn.Dropout(.10),nn.Linear(96,32),nn.ReLU())
        self.ability=nn.Linear(32,4)
        self.criteria=nn.ModuleList([nn.Linear(32,CRITERION_MAX-CRITERION_MIN) for _ in range(9)])
    def forward(self,x):
        z=self.net(x)
        return [h(z) for h in self.criteria],self.ability(z)

student=Student(latent.shape[1]).to(DEVICE)
# Teacher logits from the test set.
teacher_crit=torch.tensor([tpc[:,i] for i in range(9)],dtype=torch.float32).T.to(DEVICE)
teacher_ability=torch.tensor(tpa,dtype=torch.long,device=DEVICE)
sx=torch.tensor(latent,dtype=torch.float32,device=DEVICE)
opt_s=torch.optim.AdamW(student.parameters(),lr=1e-3,weight_decay=1e-4)
for ep in range(20):
    student.train(); opt_s.zero_grad()
    so,sa=student(sx)
    loss=0
    for i,o in enumerate(so):
        # Distill teacher hard ordinal targets plus small supervised criterion signal.
        loss=loss+F.mse_loss(torch.sigmoid(o).sum(1),teacher_crit[:,i].float())
    loss=loss/9+F.cross_entropy(sa,teacher_ability)
    loss.backward(); opt_s.step()
print("Student distillation complete. Parameters:",sum(p.numel() for p in student.parameters()))

In [ ]:
# =========================
# Lightweight deployment export
# =========================
student.eval()
example=torch.randn(1,latent.shape[1],device=DEVICE)
try:
    scripted=torch.jit.trace(student,example)
    scripted.save(str(OUT_DIR/"IHSLCC_student_distilled.pt"))
    print("TorchScript student saved.")
except Exception as e:
    print("TorchScript export warning:",e)

# Dynamic quantization is CPU-oriented.
try:
    quantized=torch.quantization.quantize_dynamic(student.cpu(),{nn.Linear},dtype=torch.qint8)
    torch.save(quantized.state_dict(),OUT_DIR/"IHSLCC_student_dynamic_int8_state.pt")
    print("Dynamic INT8 state saved.")
except Exception as e:
    print("Quantization warning:",e)

# Model-size summary
def count_params(m): return sum(p.numel() for p in m.parameters())
print("Teacher trainable parameters:",sum(p.numel() for p in model.parameters() if p.requires_grad))
print("Student parameters:",count_params(student))

In [ ]:
# =========================
# Save complete prediction table and run summary
# =========================
summary={
"rows":int(len(df)),
"train_rows":int(len(tr_idx)),
"validation_rows":int(len(va_idx)),
"test_rows":int(len(te_idx)),
"text_column":TEXT_COL,
"model":MODEL_NAME,
"device":str(DEVICE),
"ability_accuracy":float(ability_metrics.loc[0,"accuracy"]),
"ability_macro_F1":float(ability_metrics.loc[0,"macro_F1"]),
"ability_QWK":float(ability_metrics.loc[0,"QWK"]),
"hdbscan_clusters":int(n_clusters),
"hdbscan_noise_proportion":float(noise),
"hdbscan_silhouette":None if np.isnan(sil) else float(sil)
}
with open(OUT_DIR/"run_summary.json","w",encoding="utf-8") as f: json.dump(summary,f,indent=2)
print(json.dumps(summary,indent=2))
print("Outputs saved to:",OUT_DIR.resolve())